# Team 10 - Member 8: self-contained late-fusion model (CASMI 2026)

Trains **everything inside this notebook**; it needs no other member's outputs.

1. **Library search**: cosine between a query's pooled spectrum and each structure's summed training spectra.
2. **Fingerprint net**: MLP, binned spectrum -> 2048-bit Morgan fingerprint (BCE). Candidates are ranked by fingerprint log-likelihood.
3. **Late fusion**: per-query normalisation (RRF), weights tuned on two validation sets: *absent* (held-out structures) and *covered* (leave-query-out).

Candidates are the structures seen in training, filtered by precursor mass (COCONUT can be added later).
Run with a **GPU** and the competition data attached. Internet is only needed if the repo is not attached as a dataset.

In [ ]:
# 1) Find the repo code: attach it as a Kaggle Dataset (needs src/casmi inside), else clone it
# (the clone needs internet, a public repo and the code merged to the chosen branch).
import glob
import os
import subprocess
import sys

hits = glob.glob('/kaggle/input/**/src/casmi', recursive=True)
if hits:
    REPO = os.path.dirname(os.path.dirname(hits[0]))
else:
    REPO = '/kaggle/working/repo'
    if not os.path.isdir(REPO):
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'main',
                        'https://github.com/ChaitanyaUppalapati/Team-10-Enveda-CASMI-2026.git', REPO], check=True)
for p in (os.path.join(REPO, 'src'), os.path.join(REPO, 'models', 'ensembles')):
    sys.path.insert(0, p)
print('repo:', REPO)
try:
    import rdkit
    print('rdkit', rdkit.__version__)
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'rdkit'], check=True)

In [ ]:
# 2) Locate the competition files and set the config. Lower the caps for a quick dry run.
import glob

import torch
from ensemble_common.standalone import pipeline as P


def find(name):
    return next(iter(glob.glob(f'/kaggle/input/**/{name}', recursive=True)), None)


cfg = P.Config(
    train_path=find('train.parquet'), test_path=find('test.parquet'),
    # dry run: max_batches=3, max_absent_queries=200, max_covered_queries=200, max_fp_train=20000
)
print(cfg.train_path, cfg.test_path, 'cuda' if torch.cuda.is_available() else 'CPU ONLY - enable a GPU')

In [ ]:
# 3) One pass over training: structures, fingerprints, library sums, validation queries, fingerprint-net data.
st = P.build_state(cfg)
print('fp train set:', st.fp_X.shape, ' library spectra:', int(st.library.counts.sum()))

In [ ]:
# 4) Train the fingerprint network (GPU).
from ensemble_common.standalone.fpnet import TorchFingerprintModel

fp_model = TorchFingerprintModel(epochs=12, seed=cfg.seed).fit(st.fp_X, st.fp_Y)

In [ ]:
# 5) Validate: per-source MRR@25, candidate coverage, tuned fusion weights, fused MRR (absent / covered).
import json

report = P.tune_and_report(st, cfg, fp_model)
fused_val = report.pop('fused_scores')
print(json.dumps(report, indent=2))
os.makedirs('/kaggle/working/outputs', exist_ok=True)
json.dump(report, open('/kaggle/working/outputs/fusion_report.json', 'w'), indent=2)

In [ ]:
# 6) Test predictions: top-25 candidates per molecule, fused with the tuned weights.
import pandas as pd

test_scores = P.predict_test(st, cfg, fp_model, report['weights'])
final = P.attach_smiles(test_scores, st)
final.to_csv('/kaggle/working/outputs/test_top25_with_smiles.csv', index=False)
test_scores.to_parquet('/kaggle/working/outputs/test_scores.parquet', index=False)
print(final.groupby('query_id').size().describe())
final.head()

In [ ]:
# 7) submission.csv. The sample file has one row per molecule; CHECK how it stores several SMILES before submitting.
sample_path = find('sample_submission.csv')
sample = pd.read_csv(sample_path) if sample_path else None
print(sample.head() if sample is not None else 'no sample_submission.csv found')
SMILES_SEP = ';'   # <-- set to match the sample (look at a row above); default is a GUESS
rows = (final.sort_values(['query_id', 'rank']).groupby('query_id')['smiles']
        .apply(lambda s: SMILES_SEP.join(s)).reset_index().rename(columns={'query_id': 'molecule_id'}))
if sample is not None:
    missing = set(sample.molecule_id) - set(rows.molecule_id)
    print('molecules without any candidate:', len(missing))
    fill = pd.DataFrame({'molecule_id': sorted(missing), 'smiles': ''})
    rows = pd.concat([rows, fill], ignore_index=True)
rows.to_csv('/kaggle/working/submission.csv', index=False)
print(rows.shape)